# Del notebook al microservicio · Parte 1: el modelo

**ITY1102 · Arquitectura de Sistemas de IA** — Caso 0 (de referencia): *Minera Pampa Norte* · mantenimiento predictivo de camiones de extracción (CAEX)

> **Caso ficticio.** Minera Pampa Norte opera 60 camiones de extracción en tres flotas. Cada día, la telemetría de cada camión deja una lectura: horas desde la última mantención, temperatura del motor, presión de aceite, vibración, carga promedio, rpm y alertas.
> Cuando un CAEX falla en ruta, el costo es alto: grúa, reparación correctiva y producción perdida. La gerencia de mantenimiento quiere saber **qué camiones tienen alta probabilidad de fallar en los próximos 7 días** para mandarlos a inspección antes.

| Concepto | Valor (supuesto del caso) |
|---|---|
| Falla no detectada (camión se detiene en ruta) | **USD 20.000** |
| Inspección preventiva (cada alerta del modelo) | **USD 2.500** |
| Lo que pide el gerente | *"No quiero un modelo con 95% de exactitud. Quiero gastar menos en fallas sin mandar a taller a toda la flota."* |

### Dónde estamos en la arquitectura

```
 [ notebook ]  ──►  preprocesador.joblib + modelo.onnx + metadata.json  ──►  [ FastAPI + onnxruntime ]  ──►  [ imagen Docker ]  ──►  Postman
 PyTorch o TF            (artefacto versionado)                                  (parte 2)                      (parte 3)            (parte 4)
```

Esto es la pieza **"Microservicio de IA"** de la EP2: *un modelo real, entrenado por el equipo, serializado dentro de la imagen, que expone `POST /predict` y `GET /health` y responde la predicción con la versión del modelo.*

### Ruta del notebook
1. Generar los datos crudos
2. Perfilar (filas, tipos, nulos, duplicados)
3. Limpiar con reglas escritas
4. Separar entrenamiento / validación / prueba
5. Línea base "tonta" y por qué la exactitud engaña
6. Línea base clásica con scikit-learn
7. Evaluar con métricas que importan al negocio
8. **Red neuronal con PyTorch o TensorFlow**
9. **Exportar a ONNX** y verificar con `onnxruntime`
10. **Actividad: mejora la red** 🏁
11. Anexo: cómo llevar esto a tu caso semestral

### Entrenar con PyTorch o TensorFlow, servir con ONNX

Cada equipo elige **PyTorch** o **TensorFlow/Keras** para entrenar. Lo que llega a la imagen Docker no es el framework: es el modelo exportado a **ONNX** (*Open Neural Network Exchange*), un formato estándar que ejecuta `onnxruntime`.

| Para servir el modelo, la imagen instala… | Tamaño instalado (medido) | ¿Cumple la EP2? |
|---|---|---|
| `torch` desde PyPI en Linux (trae CUDA) | ≈ 4.600 MB | ✘ Prohibido: no se usa CUDA |
| `tensorflow-cpu` | ≈ 1.500 MB | Muy pesado para lo que hace |
| **`onnxruntime` + numpy** | **≈ 120 MB** | ✔ Recomendado |

**Reglas del curso para la imagen de IA**
1. Sin CUDA ni librerías de GPU: el computador de la defensa no tiene GPU, el modelo corre en CPU.
2. El framework de entrenamiento (PyTorch / TensorFlow) **no va en la imagen**: se usa en el notebook y en el job de CI.
3. La imagen sirve el modelo con `onnxruntime` (recomendado) y debe respetar el límite de tamaño del enunciado.

La elección es una **decisión de arquitectura** (IE6: *"justifica el framework del modelo"*): se entrena con la herramienta más cómoda y se sirve con el runtime más liviano.

## 0 · Preparación

In [ ]:
# PyTorch y TensorFlow ya vienen instalados en Colab.
# Agregamos: ONNX (formato), onnxruntime (para verificar como lo hará la API),
# tf2onnx (lo usa Keras para exportar) y la MISMA versión de scikit-learn que la imagen.
!pip install -q scikit-learn==1.9.1 onnx onnxruntime==1.30.0 tf2onnx

In [ ]:
import json
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

warnings.filterwarnings("ignore", category=DeprecationWarning)

FRAMEWORK = "pytorch"          # ← tu equipo elige: "pytorch" o "tensorflow"
SEMILLA = 42                   # reproducibilidad: mismo resultado en cualquier computador
CARPETA = Path("/content")     # disco temporal de Colab
RUTA_DATOS = CARPETA / "camiones_crudo.csv"
RUTA_MODELO = CARPETA / "model"   # aquí se exporta preprocesador.joblib + modelo.onnx + metadata.json

print(f"Framework elegido: {FRAMEWORK}")
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

## 1 · Generar los datos crudos

Los datos son **sintéticos**: los genera la celda siguiente. Siguen una "física" escondida (umbrales, interacciones, curvas en U) que el modelo tiene que descubrir. Igual que en sus casos semestrales, vienen **sucios a propósito**: duplicados, nulos, categorías mal escritas y sensores con códigos de error.

> No es necesario entender cada línea del generador. Lo importante es lo que produce: un CSV como los que vienen en el zip de su caso.

In [ ]:
FLOTAS = {"A": (290, 0.00), "B": (330, 0.15), "C": (360, 0.55)}   # flota: (capacidad t, sesgo de riesgo)

def _verdad_oculta(df, rng):
    # La "física" que el modelo tiene que descubrir (no lineal a propósito)
    sesgo_flota = df["flota"].map({k: v[1] for k, v in FLOTAS.items()}).to_numpy()
    capacidad = df["flota"].map({k: v[0] for k, v in FLOTAS.items()}).to_numpy()
    sobrecarga = np.clip(df["carga_promedio_ton"].to_numpy() / capacidad - 1.0, 0, None)
    logit = (
        -8.2
        + 0.006 * df["horas_desde_mantencion"].to_numpy()
        + 0.55 * np.clip(df["temp_motor_c"].to_numpy() - 93, 0, None)
        + 0.40 * np.clip(48 - df["presion_aceite_psi"].to_numpy(), 0, None)
        + 1.80 * np.clip(df["vibracion_mm_s"].to_numpy() - 6, 0, None)
        + 30.0 * sobrecarga * (df["temp_motor_c"].to_numpy() > 90)
        + 3.0e-5 * (df["rpm_promedio"].to_numpy() - 1650) ** 2
        + 0.60 * df["alertas_24h"].to_numpy()
        + 0.08 * df["antiguedad_anios"].to_numpy()
        + 0.30 * (df["turno"].to_numpy() == "noche")
        + sesgo_flota
    )
    prob = 1 / (1 + np.exp(-logit))
    return (rng.random(len(df)) < prob).astype(int)

def _ensuciar(df, rng):
    df = df.copy()
    n = len(df)
    df["falla_7d"] = df["falla_7d"].astype("Int64")
    # 1. Categorías escritas de distintas formas
    variantes_turno = {"dia": ["Dia", "DIA", "día", "dia "], "noche": ["Noche", "NOCHE", " noche"]}
    for valor, variantes in variantes_turno.items():
        mask = (df["turno"] == valor) & (rng.random(n) < 0.12)
        df.loc[mask, "turno"] = rng.choice(variantes, size=mask.sum())
    mask = rng.random(n) < 0.05
    df.loc[mask, "flota"] = df.loc[mask, "flota"].str.lower() + " "
    # 2. Nulos en sensores (el sensor no reportó)
    for col, tasa in [("temp_motor_c", 0.015), ("presion_aceite_psi", 0.02), ("vibracion_mm_s", 0.01)]:
        df.loc[rng.random(n) < tasa, col] = np.nan
    # 3. Lecturas fuera de rango (códigos de error del sensor)
    df.loc[rng.random(n) < 0.004, "temp_motor_c"] = -999.0
    df.loc[rng.random(n) < 0.003, "presion_aceite_psi"] = -1.0
    df.loc[rng.random(n) < 0.003, "carga_promedio_ton"] = 9999.0
    # 4. Etiqueta faltante
    df.loc[rng.random(n) < 0.004, "falla_7d"] = pd.NA
    # 5. Filas duplicadas (el ETL reenvió el lote)
    dup = df.sample(n=int(n * 0.012), random_state=int(rng.integers(0, 10_000)))
    df = pd.concat([df, dup], ignore_index=True)
    return df.sample(frac=1, random_state=1).reset_index(drop=True)

def generar(filas=6000, semilla=SEMILLA):
    rng = np.random.default_rng(semilla)
    n_camiones = 60
    camiones = pd.DataFrame({
        "id_camion": [f"CAEX-{i:03d}" for i in range(1, n_camiones + 1)],
        "flota": rng.choice(list(FLOTAS), size=n_camiones, p=[0.4, 0.35, 0.25]),
        "antiguedad_anios": rng.integers(1, 15, size=n_camiones),
    })
    idx = rng.integers(0, n_camiones, size=filas)
    df = camiones.iloc[idx].reset_index(drop=True)
    capacidad = df["flota"].map({k: v[0] for k, v in FLOTAS.items()}).to_numpy()
    fechas = pd.Timestamp("2026-03-01") + pd.to_timedelta(rng.integers(0, 180, filas), unit="D")
    df.insert(0, "id_lectura", np.arange(100001, 100001 + filas))
    df["fecha"] = fechas.strftime("%Y-%m-%d")
    df["turno"] = rng.choice(["dia", "noche"], size=filas)
    df["horas_desde_mantencion"] = rng.gamma(2.2, 110, filas).round(0)
    df["carga_promedio_ton"] = (capacidad * rng.normal(0.97, 0.06, filas)).round(1)
    df["temp_motor_c"] = (rng.normal(86, 5.5, filas) + 0.012 * df["horas_desde_mantencion"].to_numpy()).round(1)
    df["presion_aceite_psi"] = (rng.normal(56, 6, filas) - 0.010 * df["horas_desde_mantencion"].to_numpy()).round(1)
    df["vibracion_mm_s"] = rng.lognormal(np.log(4.2), 0.32, filas).round(2)
    df["rpm_promedio"] = rng.normal(1650, 170, filas).round(0)
    df["alertas_24h"] = rng.poisson(0.35 + 0.0012 * df["horas_desde_mantencion"].to_numpy())
    df["falla_7d"] = _verdad_oculta(df, rng)
    columnas = ["id_lectura", "id_camion", "fecha", "flota", "turno", "antiguedad_anios",
                "horas_desde_mantencion", "carga_promedio_ton", "temp_motor_c",
                "presion_aceite_psi", "vibracion_mm_s", "rpm_promedio", "alertas_24h", "falla_7d"]
    return _ensuciar(df[columnas], rng)

generar().to_csv(RUTA_DATOS, index=False)
crudo = pd.read_csv(RUTA_DATOS)
crudo.head()

## 2 · Perfilado

Antes de tocar un dato, se mide. Estas cifras van al capítulo **8.4 Datos** del arc42 (*"perfilado antes y después"*).

In [ ]:
def perfilar(df: pd.DataFrame, nombre: str) -> pd.DataFrame:
    print(f"{nombre}: {df.shape[0]:,} filas × {df.shape[1]} columnas | duplicados exactos: {df.duplicated().sum()}")
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "% nulos": (df.isna().mean() * 100).round(2),
        "únicos": df.nunique(),
    })

perfil_antes = perfilar(crudo, "ANTES")
perfil_antes

In [ ]:
# Las categorías: ¿cuántas formas distintas de escribir lo mismo?
print(crudo["turno"].value_counts(dropna=False), "\n")
print(crudo["flota"].value_counts(dropna=False))

In [ ]:
# Rangos físicos: ¿hay lecturas imposibles?
crudo[["temp_motor_c", "presion_aceite_psi", "carga_promedio_ton", "vibracion_mm_s"]].describe().T

**Lo que encontramos** (anótalo: en la defensa preguntan *"¿cuántas filas descartó tu limpieza y por qué?"*):

- Filas duplicadas (el ETL reenvió lotes).
- `turno` y `flota` escritos de varias formas (`DIA`, `día`, ` noche`, `b `…).
- Nulos en sensores (el sensor no reportó).
- Valores imposibles: temperatura `-999`, presión `-1`, carga `9999` → códigos de error del sensor.
- Etiqueta `falla_7d` vacía en algunas filas.

## 3 · Limpieza con reglas escritas

Cada problema tiene **una regla**, y cada regla decide si la fila se **corrige**, se **descarta** o se **marca**.

| Regla | Problema | Decisión | Por qué |
|---|---|---|---|
| R1 | Duplicados por `id_lectura` | Descartar | Es la misma lectura dos veces; inflaría el entrenamiento |
| R2 | Categorías mal escritas | Corregir | `DIA` y `día` son lo mismo; normalizamos a `dia`/`noche` y `A`/`B`/`C` |
| R3 | Sensor fuera de rango físico | Marcar como nulo | El resto de la fila sirve; el nulo lo imputa el modelo |
| R4 | Etiqueta faltante | Descartar | Sin etiqueta no se puede entrenar ni evaluar |

In [ ]:
RANGOS = {                                  # rango físico plausible de cada sensor
    "temp_motor_c": (40, 140),
    "presion_aceite_psi": (0, 120),
    "carga_promedio_ton": (0, 450),
    "vibracion_mm_s": (0, 50),
}

def limpiar(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    df = df.copy()
    registro = {"filas_entrada": len(df)}

    # R1 · duplicados
    antes = len(df)
    df = df.drop_duplicates(subset="id_lectura")
    registro["R1_duplicados_descartados"] = antes - len(df)

    # R2 · normalizar categorías
    df["turno"] = (df["turno"].str.strip().str.lower()
                   .str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii"))
    df["flota"] = df["flota"].str.strip().str.upper()

    # R3 · fuera de rango → nulo (se marca, no se descarta)
    for col, (lo, hi) in RANGOS.items():
        fuera = df[col].notna() & ~df[col].between(lo, hi)
        registro[f"R3_{col}_fuera_de_rango"] = int(fuera.sum())
        df.loc[fuera, col] = np.nan

    # R4 · etiqueta faltante
    antes = len(df)
    df = df.dropna(subset=["falla_7d"])
    df["falla_7d"] = df["falla_7d"].astype(int)
    registro["R4_sin_etiqueta_descartadas"] = antes - len(df)

    registro["filas_salida"] = len(df)
    return df, registro

datos, registro_limpieza = limpiar(crudo)
pd.Series(registro_limpieza, name="cifra")

In [ ]:
perfil_despues = perfilar(datos, "DESPUÉS")
print(datos["turno"].value_counts().to_dict(), datos["flota"].value_counts().to_dict())
perfil_despues

**Ojo:** los nulos de sensores **no** se rellenan aquí con el promedio de todo el dataset. Eso lo hace el **preprocesador** (secciones 6 y 8), que aprende las medianas solo con datos de entrenamiento. Dos razones:
1. Evita **fuga de información** (*data leakage*) del conjunto de prueba.
2. La API va a recibir lecturas con sensores en `null`, y el preprocesador serializado ya sabe cómo imputarlas. **La limpieza que vive en el preprocesador viaja con el modelo.**

### Exploración rápida

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

datos["falla_7d"].value_counts().sort_index().plot.bar(ax=ax[0], color=["#4C78A8", "#E45756"])
ax[0].set_title("Balance de clases"); ax[0].set_xticklabels(["sin falla", "falla 7d"], rotation=0)

for clase, color in [(0, "#4C78A8"), (1, "#E45756")]:
    datos.loc[datos.falla_7d == clase, "temp_motor_c"].plot.hist(ax=ax[1], bins=40, alpha=.6, color=color, density=True)
ax[1].set_title("Temperatura del motor por clase"); ax[1].legend(["sin falla", "falla"])

datos.groupby("flota")["falla_7d"].mean().plot.bar(ax=ax[2], color="#72B7B2")
ax[2].set_title("Tasa de falla por flota"); ax[2].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

print(f"Tasa de falla: {datos.falla_7d.mean():.1%}")

**Pregunta para la sala:** solo ~14% de las lecturas termina en falla. Si un "modelo" respondiera siempre *"no va a fallar"*, ¿qué exactitud tendría?

## 4 · Variables de entrada y separación en tres conjuntos

Antes de entrenar, repartimos las filas en **tres grupos que no se mezclan**. Piensa en cómo te preparas para una prueba:

| Conjunto | % | Filas | Se parece a… | Para qué lo usamos |
|---|---|---|---|---|
| **Entrenamiento** | 60% | ≈ 3.600 | La guía de ejercicios | El modelo **aprende** aquí: la estudia todas las veces que quiera |
| **Validación** | 15% | ≈ 900 | Los ensayos | Nosotros **decidimos** aquí: umbral, número de épocas, tamaño de la red |
| **Prueba** | 25% | ≈ 1.500 | El examen final | Se **informa** la nota. Se mira **una sola vez**, al final |

**¿Por qué no basta con entrenamiento y prueba?** Cada vez que miramos un resultado y cambiamos algo (más épocas, otro umbral), el modelo queda un poco ajustado a esos datos. Si esas decisiones se tomaran con prueba, sería como estudiar con las preguntas del examen: la nota final saldría inflada. Validación absorbe esas decisiones y prueba sigue siendo un examen limpio.

**Cómo se hace:** `train_test_split` solo parte en dos, así que se usa dos veces:

```
5.977 filas ──► 25% prueba (bajo llave)
            └─► 75% resto ──► 20% validación
                          └─► 80% entrenamiento
```

**`stratify`** asegura que los tres grupos tengan la misma proporción de fallas (~14%), y **`random_state`** que todos obtengamos exactamente el mismo reparto.

> Regla de oro: la separación se hace **antes** de ajustar cualquier cosa. Si algo de prueba influye en el entrenamiento o en una decisión, hay **fuga de información** (*data leakage*).

In [ ]:
from sklearn.model_selection import train_test_split

VARIABLES_NUMERICAS = [
    "antiguedad_anios", "horas_desde_mantencion", "carga_promedio_ton", "temp_motor_c",
    "presion_aceite_psi", "vibracion_mm_s", "rpm_promedio", "alertas_24h",
]
VARIABLES_CATEGORICAS = ["flota", "turno"]
VARIABLES = VARIABLES_NUMERICAS + VARIABLES_CATEGORICAS
OBJETIVO = "falla_7d"

X, y = datos[VARIABLES], datos[OBJETIVO]

X_resto, X_test, y_resto, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEMILLA)
X_train, X_val, y_train, y_val = train_test_split(X_resto, y_resto, test_size=0.20, stratify=y_resto, random_state=SEMILLA)

for nombre, (Xs, ys) in {"Entrenamiento": (X_train, y_train), "Validación": (X_val, y_val), "Prueba": (X_test, y_test)}.items():
    print(f"{nombre:<14} {len(Xs):>5,} filas ({ys.mean():.1%} fallas)")

> `id_lectura`, `id_camion` y `fecha` **no** entran al modelo. Son identificadores: el modelo memorizaría camiones en vez de aprender patrones, y no generalizaría a un camión nuevo.

## 5 · La línea base "tonta"

Antes de cualquier modelo real, se mide lo que logra **no hacer nada**. Todo modelo tiene que ganarle a esto.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

tonto = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
pred_tonto = tonto.predict(X_test)
print(f"Exactitud (accuracy): {accuracy_score(y_test, pred_tonto):.1%}")
print(f"Fallas detectadas (recall): {recall_score(y_test, pred_tonto):.1%}")

**~86% de exactitud sin detectar ni una falla.** Por eso en datos desbalanceados la exactitud no sirve como métrica principal (es la misma trampa del caso de fraude: *"no me traigan un modelo con 99% de exactitud"*).

Vamos a medir con lo que le importa al negocio:

In [ ]:
COSTO_FALLA_NO_DETECTADA = 20_000   # USD, falso negativo
COSTO_INSPECCION = 2_500            # USD, cada alerta (verdadera o falsa) manda el camión a inspección

def costo_operacional(y_real, y_pred) -> int:
    y_real, y_pred = np.asarray(y_real), np.asarray(y_pred)
    fallas_no_detectadas = int(((y_pred == 0) & (y_real == 1)).sum())
    inspecciones = int((y_pred == 1).sum())
    return fallas_no_detectadas * COSTO_FALLA_NO_DETECTADA + inspecciones * COSTO_INSPECCION

print(f"Sin modelo (nunca inspeccionar):  USD {costo_operacional(y_test, np.zeros_like(y_test)):>10,}")
print(f"Inspeccionar toda la flota:       USD {costo_operacional(y_test, np.ones_like(y_test)):>10,}")

## 6 · Línea base clásica: regresión logística (scikit-learn)

Antes de la red neuronal, un modelo clásico y rápido. Sirve de **vara**: si la red no le gana, no se justifica su complejidad.

El **preprocesamiento** (imputar nulos, escalar, one-hot) se define en una función porque lo vamos a reutilizar con la red.

```
 lectura ──► [ imputar nulos → escalar ]  numéricas    ──►  vector de 13 números  ──►  modelo  ──►  probabilidad
             [ one-hot ]                  categóricas
```

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def construir_preprocesador() -> ColumnTransformer:
    return ColumnTransformer([
        ("num", Pipeline([
            ("imputar", SimpleImputer(strategy="median")),   # sensor en null → mediana de entrenamiento
            ("escalar", StandardScaler()),
        ]), VARIABLES_NUMERICAS),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), VARIABLES_CATEGORICAS),
    ])

modelo_base = Pipeline([
    ("prep", construir_preprocesador()),
    ("clf", LogisticRegression(max_iter=1000)),
])
modelo_base.fit(X_train, y_train)
modelo_base

## 7 · Evaluación

In [ ]:
from sklearn.metrics import (ConfusionMatrixDisplay, average_precision_score,
                             classification_report, recall_score, roc_auc_score)

prob_test = modelo_base.predict_proba(X_test)[:, 1]   # probabilidad de falla
pred_test = (prob_test >= 0.5).astype(int)              # umbral por defecto

print(classification_report(y_test, pred_test, target_names=["sin falla", "falla"], digits=3))
print(f"ROC-AUC: {roc_auc_score(y_test, prob_test):.3f}  |  PR-AUC: {average_precision_score(y_test, prob_test):.3f}")
print(f"Costo operacional (umbral 0,5): USD {costo_operacional(y_test, pred_test):,}")

ConfusionMatrixDisplay.from_predictions(y_test, pred_test, display_labels=["sin falla", "falla"], cmap="Blues")
plt.title("Línea base · umbral 0,5"); plt.show()

**Cómo leerlo:**
- **Recall de "falla"**: de las fallas reales, ¿cuántas detectó? Es lo que más cuesta perder.
- **Precision de "falla"**: de las alertas, ¿cuántas eran fallas de verdad? Cada alerta falsa es una inspección pagada de más.
- **ROC-AUC / PR-AUC**: qué tan bien *ordena* el modelo por riesgo, independiente del umbral. PR-AUC es más honesta con clases desbalanceadas.

### El umbral es una decisión de negocio, no del algoritmo
El `0,5` es arbitrario. Con una falla 8 veces más cara que una inspección, conviene alertar con probabilidades bastante menores. Veamos cómo cambia el costo:

In [ ]:
umbrales = np.arange(0.05, 0.96, 0.01)
costos = [costo_operacional(y_test, (prob_test >= u).astype(int)) for u in umbrales]

plt.figure(figsize=(8, 3.5))
plt.plot(umbrales, costos, color="#4C78A8")
plt.axvline(0.5, ls="--", color="gray", label="umbral 0,5")
plt.xlabel("umbral de decisión"); plt.ylabel("costo USD"); plt.title("Costo operacional según umbral (modelo base)")
plt.legend(); plt.show()

> ⚠️ Ese gráfico usa **prueba** solo para *mostrar* la idea. Elegir el umbral mirando prueba es hacer trampa. Desde ahora el umbral se elige con **validación**, y cada modelo queda registrado en una **tabla de posiciones** con su costo en prueba.

In [ ]:
tabla = []

def mejor_umbral(y_real, prob) -> float:
    candidatos = np.arange(0.02, 0.96, 0.01)
    costos = [costo_operacional(y_real, (prob >= u).astype(int)) for u in candidatos]
    return float(candidatos[int(np.argmin(costos))])

def registrar(nombre: str, prob_val, prob_test) -> float:
    umbral = mejor_umbral(y_val, prob_val)                # ← decidido SIN mirar prueba
    pred = (prob_test >= umbral).astype(int)
    tabla.append({
        "modelo": nombre, "umbral": round(umbral, 2),
        "costo_usd": costo_operacional(y_test, pred),
        "recall": round(recall_score(y_test, pred), 3),
        "alertas": int(pred.sum()),
        "roc_auc": round(roc_auc_score(y_test, prob_test), 3),
        "pr_auc": round(average_precision_score(y_test, prob_test), 3),
    })
    display(pd.DataFrame(tabla).sort_values("costo_usd").reset_index(drop=True))
    return umbral

umbral_lr = registrar("0 · LogReg (scikit-learn)", modelo_base.predict_proba(X_val)[:, 1], prob_test)

## 8 · La red neuronal (PyTorch o TensorFlow)

La red no sabe de nulos ni de textos como `"noche"`: recibe un **vector de números**. Por eso separamos el trabajo en dos piezas que viajan juntas a la imagen:

```
 JSON ──► preprocesador.joblib ──► vector de 13 números ──► modelo.onnx ──► probabilidad
          (scikit-learn)                                    (red de PyTorch o TF, ejecutada por onnxruntime)
          imputar · escalar · one-hot
```

Primero ajustamos el preprocesador **solo con entrenamiento** y transformamos los tres conjuntos a matrices `float32` (el tipo que esperan las redes).

In [ ]:
prep = construir_preprocesador().fit(X_train)

A_train = prep.transform(X_train).astype(np.float32)
A_val   = prep.transform(X_val).astype(np.float32)
A_test  = prep.transform(X_test).astype(np.float32)

print("Forma de A_train:", A_train.shape)
print("Columnas que ve la red:", list(prep.get_feature_names_out()))

### Opción A · PyTorch

Una red **MLP** (perceptrón multicapa) con dos capas ocultas. Lo importante del código:
- `nn.Sequential`: capas `Linear → ReLU → Dropout`, y una salida de 1 neurona.
- `BCEWithLogitsLoss(pos_weight=…)`: pérdida para clasificación binaria que **pesa más las fallas** (son solo 14%).
- Entrenamiento por **mini-lotes** de 64 filas, con Adam.
- Cada 10 épocas mostramos la PR-AUC en **validación** para ver si sigue aprendiendo o empieza a sobreajustar.

In [ ]:
def entrenar_red_pytorch(capas=(64, 32), dropout=0.2, epocas=60, lr=1e-3, peso_clase=True):
    import torch
    import torch.nn as nn

    torch.manual_seed(SEMILLA)
    bloques, entrada = [], A_train.shape[1]
    for n in capas:
        bloques += [nn.Linear(entrada, n), nn.ReLU(), nn.Dropout(dropout)]
        entrada = n
    red = nn.Sequential(*bloques, nn.Linear(entrada, 1))         # salida: 1 logit

    peso = (y_train == 0).sum() / (y_train == 1).sum() if peso_clase else 1.0
    perdida = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([peso], dtype=torch.float32))
    optim = torch.optim.Adam(red.parameters(), lr=lr, weight_decay=1e-4)
    X = torch.from_numpy(A_train)
    Y = torch.tensor(y_train.to_numpy(), dtype=torch.float32).unsqueeze(1)
    gen = torch.Generator().manual_seed(SEMILLA)

    def predecir(A):
        red.eval()
        with torch.no_grad():
            return torch.sigmoid(red(torch.from_numpy(A))).numpy().ravel()

    for epoca in range(1, epocas + 1):
        red.train()
        orden = torch.randperm(len(X), generator=gen)
        for i in range(0, len(X), 64):                           # mini-lotes de 64
            idx = orden[i:i + 64]
            optim.zero_grad()
            perdida(red(X[idx]), Y[idx]).backward()
            optim.step()
        if epoca % 10 == 0:
            print(f"época {epoca:>3} · PR-AUC validación {average_precision_score(y_val, predecir(A_val)):.3f}")

    class ConProbabilidad(nn.Module):        # el ONNX entregará probabilidad (0 a 1), no logit
        def __init__(self, m):
            super().__init__()
            self.m = m
        def forward(self, x):
            return torch.sigmoid(self.m(x))

    def exportar_onnx(ruta):
        torch.onnx.export(ConProbabilidad(red).eval(), torch.zeros(1, A_train.shape[1]), str(ruta),
                          input_names=["x"], output_names=["prob"],
                          dynamic_axes={"x": {0: "n"}, "prob": {0: "n"}},   # n = filas variables (lotes)
                          opset_version=17, dynamo=False)

    return {"predecir": predecir, "exportar_onnx": exportar_onnx, "framework": f"torch {torch.__version__}"}

### Opción B · TensorFlow / Keras

La misma red con Keras. Keras ya incluye la `sigmoid` en la última capa, y para pesar más las fallas se usa `class_weight`. Se entrena de a 10 épocas para mostrar la PR-AUC en validación igual que en PyTorch.

In [ ]:
def entrenar_red_tensorflow(capas=(64, 32), dropout=0.2, epocas=60, lr=1e-3, peso_clase=True):
    import os
    os.environ.setdefault("KERAS_BACKEND", "tensorflow")
    import keras
    import tensorflow as tf

    keras.utils.set_random_seed(SEMILLA)
    red = keras.Sequential([keras.Input(shape=(A_train.shape[1],))])
    for n in capas:
        red.add(keras.layers.Dense(n, activation="relu"))
        red.add(keras.layers.Dropout(dropout))
    red.add(keras.layers.Dense(1, activation="sigmoid"))          # salida: probabilidad
    red.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy")

    peso = float((y_train == 0).sum() / (y_train == 1).sum()) if peso_clase else 1.0

    def predecir(A):
        return red.predict(A, verbose=0).ravel()

    hechas = 0
    while hechas < epocas:
        tramo = min(10, epocas - hechas)
        red.fit(A_train, y_train.to_numpy(), epochs=tramo, batch_size=64,
                class_weight={0: 1.0, 1: peso}, verbose=0)
        hechas += tramo
        print(f"época {hechas:>3} · PR-AUC validación {average_precision_score(y_val, predecir(A_val)):.3f}")

    def exportar_onnx(ruta):
        red.export(str(ruta), format="onnx")                      # usa tf2onnx por debajo

    return {"predecir": predecir, "exportar_onnx": exportar_onnx, "framework": f"tensorflow {tf.__version__}"}

In [ ]:
entrenar_red = entrenar_red_pytorch if FRAMEWORK == "pytorch" else entrenar_red_tensorflow

t0 = time.perf_counter()
red = entrenar_red(capas=(64, 32), dropout=0.2, epocas=60, lr=1e-3)
print(f"Entrenada con {red['framework']} en {time.perf_counter() - t0:.1f} s")

umbral_red = registrar("1 · Red 64-32", red["predecir"](A_val), red["predecir"](A_test))

## 9 · Exportar a ONNX y verificar

Se exportan **tres archivos** a la carpeta `model/`:

| Archivo | Contenido | Quién lo usa en la API |
|---|---|---|
| `preprocesador.joblib` | Imputar, escalar y one-hot (scikit-learn) | Transforma el JSON en el vector de 13 números |
| `modelo.onnx` | La red entrenada (pesos + grafo) | `onnxruntime` calcula la probabilidad |
| `metadata.json` | Versión, framework, umbral, métricas, versión de scikit-learn | `/predict` responde la versión; `/modelo` la expone |

Antes de dar por bueno el ONNX lo **verificamos**: `onnxruntime` tiene que dar las mismas probabilidades que la red original.

In [ ]:
import onnxruntime as ort

def exportar(red, version: str, umbral: float, descripcion: str) -> dict:
    RUTA_MODELO.mkdir(parents=True, exist_ok=True)
    for viejo in ["modelo.joblib", "modelo.onnx", "preprocesador.joblib"]:
        (RUTA_MODELO / viejo).unlink(missing_ok=True)

    joblib.dump(prep, RUTA_MODELO / "preprocesador.joblib")
    red["exportar_onnx"](RUTA_MODELO / "modelo.onnx")

    # Verificación: onnxruntime (lo que usará la API) vs. la red original
    sesion = ort.InferenceSession(str(RUTA_MODELO / "modelo.onnx"), providers=["CPUExecutionProvider"])
    p_onnx = sesion.run(None, {sesion.get_inputs()[0].name: A_test})[0].ravel()
    diferencia = float(np.abs(p_onnx - red["predecir"](A_test)).max())
    assert diferencia < 1e-4, f"El ONNX no reproduce a la red (diferencia {diferencia})"

    pred = (p_onnx >= umbral).astype(int)
    metadata = {
        "nombre": "falla-caex-7d",
        "version": version,
        "formato": "onnx",
        "framework": red["framework"],
        "descripcion": descripcion,
        "entrenado_en": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "sklearn_version": sklearn.__version__,
        "variables_numericas": VARIABLES_NUMERICAS,
        "variables_categoricas": VARIABLES_CATEGORICAS,
        "umbral": round(float(umbral), 4),
        "filas_entrenamiento": int(len(X_train)),
        "metricas_prueba": {
            "roc_auc": round(roc_auc_score(y_test, p_onnx), 4),
            "pr_auc": round(average_precision_score(y_test, p_onnx), 4),
            "recall": round(recall_score(y_test, pred), 4),
            "costo_usd": costo_operacional(y_test, pred),
        },
        "limpieza": registro_limpieza,
    }
    (RUTA_MODELO / "metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")

    print(f"✔ ONNX verificado: diferencia máxima con la red original {diferencia:.1e}")
    for f in sorted(RUTA_MODELO.iterdir()):
        print(f"   {f.name:<22} {f.stat().st_size / 1024:>6.1f} KB")
    print(f"✔ Exportado {version} ({red['framework']})")
    return metadata

meta = exportar(red, version=f"1.0.0-{FRAMEWORK}", umbral=umbral_red,
                descripcion="MLP 64-32, dropout 0.2, 60 epocas")
meta["metricas_prueba"]

### Simular lo que hará la API
La API no tiene PyTorch ni TensorFlow: solo `onnxruntime`. Probemos el artefacto **exactamente** como lo usará: cargar los dos archivos desde disco y predecir un JSON con un sensor en `null`.

In [ ]:
prep_api = joblib.load(RUTA_MODELO / "preprocesador.joblib")
sesion_api = ort.InferenceSession(str(RUTA_MODELO / "modelo.onnx"), providers=["CPUExecutionProvider"])
umbral_api = json.loads((RUTA_MODELO / "metadata.json").read_text())["umbral"]

lectura = {                                  # ← esto es lo que llegará en el body de POST /predict
    "antiguedad_anios": 11, "horas_desde_mantencion": 620, "carga_promedio_ton": 395,
    "temp_motor_c": 99.5, "presion_aceite_psi": 41.0, "vibracion_mm_s": None,
    "rpm_promedio": 1980, "alertas_24h": 3, "flota": "C", "turno": "noche",
}

t0 = time.perf_counter()
x = prep_api.transform(pd.DataFrame([lectura])).astype(np.float32)
p = float(sesion_api.run(None, {sesion_api.get_inputs()[0].name: x})[0].ravel()[0])
latencia_ms = (time.perf_counter() - t0) * 1000
print(f"Probabilidad de falla en 7 días: {p:.1%}  →  {'INSPECCIONAR' if p >= umbral_api else 'operar normal'}")
print(f"Latencia de inferencia: {latencia_ms:.2f} ms")

In [ ]:
# Colab borra /content al cerrar la sesión: descarga el modelo para llevarlo al repositorio
import shutil
shutil.make_archive("/content/modelo", "zip", RUTA_MODELO)
try:
    from google.colab import files
    files.download("/content/modelo.zip")
except ImportError:
    print("No estás en Colab: el zip quedó en", "/content/modelo.zip")

---
## 10 · 🏁 Actividad: mejora la red (15 minutos, en parejas)

**Objetivo:** bajar el **costo operacional en prueba** respecto de la red base. Al final cada pareja dice su costo, su recall, su framework y su versión, y lo anotamos en la pizarra.

**Reglas del juego**
1. Las decisiones se toman mirando **validación** (la PR-AUC que se imprime cada 10 épocas). Prueba solo se informa.
2. Todo lo que cambies tiene que poder exportarse a ONNX.
3. Si mejoras, exportas con una versión nueva (`1.1.0-<tus-iniciales>`) y la API la servirá en la parte 2.

**Retos**

| # | Reto | Qué cambiar en `probar(...)` | Qué mirar |
|---|---|---|---|
| 1 | Más capacidad | `capas=(128, 64)` o `(128, 64, 32)` | ¿Mejora o solo tarda más? |
| 2 | Sobreajuste | `epocas=200, dropout=0.0` y luego `dropout=0.3` | ¿La PR-AUC de validación sube y después baja? |
| 3 | Desbalance | `peso_clase=False` | ¿Qué le pasa al umbral elegido? |
| 4 | Tasa de aprendizaje | `lr=1e-2` y `lr=1e-4` | ¿Aprende más rápido, más lento o se vuelve inestable? |
| ★ | Otro framework | Cambia `FRAMEWORK` y vuelve a ejecutar desde la celda de entrenamiento | ¿Dan lo mismo PyTorch y TensorFlow? |
| ★ | Otro modelo | `RandomForestClassifier` de scikit-learn | ¿Vale la pena una red para datos tabulares? |

In [ ]:
def probar(nombre: str, **config):
    t0 = time.perf_counter()
    r = entrenar_red(**config)
    u = registrar(nombre, r["predecir"](A_val), r["predecir"](A_test))
    print(f"({r['framework']}, {time.perf_counter() - t0:.1f} s)")
    return r, u

# Ejemplo: r1, u1 = probar("2 · Red 128-64", capas=(128, 64))

### ✍️ Tu turno · Retos 1 y 2

In [ ]:
# TODO reto 1: más capacidad
# r1, u1 = probar("2 · Red 128-64", capas=(128, 64))

# TODO reto 2: sobreajuste (mira cómo cambia la PR-AUC de validación época a época)
# r2, u2 = probar("3 · 200 épocas sin dropout", epocas=200, dropout=0.0)

### ✍️ Retos 3 y 4

In [ ]:
# TODO reto 3: sin peso de clase
# r3, u3 = probar("5 · sin peso de clase", peso_clase=False)

# TODO reto 4: tasa de aprendizaje
# r4, u4 = probar("6 · lr 0.01", lr=1e-2)

### ★ Bonus · ¿vale la pena una red para datos tabulares?
Un RandomForest con el mismo preprocesador. Si gana, la discusión es de arquitectura: ¿qué es más fácil de servir y explicar?

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# TODO bonus: entrena un RandomForest con el preprocesador y regístralo
# rf = Pipeline([("prep", construir_preprocesador()),
#                ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
#                                               class_weight="balanced_subsample", n_jobs=-1, random_state=SEMILLA))])
# rf.fit(X_train, y_train)
# registrar("7 · RandomForest", rf.predict_proba(X_val)[:, 1], rf.predict_proba(X_test)[:, 1])
# Esperado: ≈ 1.542.500, el menor costo de la tabla, con menor ROC-AUC que la red (0.931 vs 0.940)

### ✍️ Exporta tu mejor red
Cuando tengas una que le gane a la red base, expórtala con **su** umbral y una versión nueva. Esa versión es la que vas a ver en la respuesta de la API.

In [ ]:
# TODO: reemplaza r1 / u1 por tu mejor red
# meta = exportar(r1, version="1.1.0-XX", umbral=u1, descripcion="...")
# meta["metricas_prueba"]

### Para discutir antes de pasar a la API
1. ¿El modelo con mejor ROC-AUC es también el de menor costo? ¿Por qué podría no serlo?
2. Entrenaron con PyTorch (o TensorFlow), pero la imagen no lo tiene instalado. ¿Qué pasaría con el tamaño de la imagen y con el `mem_limit` si lo tuviera?
3. Si mañana una falla cuesta USD 40.000, ¿qué cambia: la red o el umbral? ¿Hay que reconstruir la imagen?
4. ¿Por qué scikit-learn sigue fijado a una versión exacta y PyTorch no?

---
## 11 · Anexo: del notebook al script, y a tu caso semestral

### El notebook no va a producción
La EP2 pide un **script versionado** que el pipeline de CI pueda ejecutar, con un **umbral mínimo de métrica**. En el repositorio es `servicio-ia/entrenar.py`, con el mismo framework que elijas:

```
python servicio-ia/entrenar.py --framework pytorch    --pr-auc-minimo 0.55
python servicio-ia/entrenar.py --framework tensorflow --capas 128,64 --version 1.1.0-xx
```

PyTorch o TensorFlow se instalan en el **job de CI** (versión CPU), nunca en la imagen.

### Cómo se traduce a cada caso

| Caso | Qué predice el modelo (EP2) | Preprocesador → entrada de la red | Qué se parece a este notebook |
|---|---|---|---|
| **1 · Recomendación e-commerce** | Propensión de compra de un usuario en una categoría | Compras y vistas por categoría, recencia, ticket → escalar / one-hot | Binaria con probabilidad; la métrica de negocio es ticket/conversión |
| **2 · Chatbot telco** | Intención de un mensaje | **Texto** → `TfidfVectorizer` → vector | Mismo esquema preprocesador + red; salida **multiclase** (softmax) |
| **3 · Fraude** | Probabilidad de fraude de una transacción | Monto, hora, dispositivo, lista negra → escalar / one-hot | **Casi idéntico**: desbalance, costo FP vs FN, umbral por costo |

In [ ]:
# Caso 2 (texto → intención): el preprocesador es TF-IDF y la salida es multiclase.
# Aquí con scikit-learn para que sea corto; con PyTorch/TF la red recibiría el vector TF-IDF
# y tendría una neurona de salida por intención (softmax).
from sklearn.feature_extraction.text import TfidfVectorizer

ejemplo_texto = pd.DataFrame({
    "mensaje": ["cuanto debo este mes", "no tengo internet desde ayer", "quiero cambiar mi plan",
                "donde viene el tecnico", "quiero anular el servicio", "se cayo la fibra"],
    "intencion": ["deuda", "falla_servicio", "cambio_plan", "visita_tecnica", "anulacion", "falla_servicio"],
})
tfidf = TfidfVectorizer(ngram_range=(1, 2), strip_accents="unicode").fit(ejemplo_texto["mensaje"])
print("Tamaño del vector de entrada de la red:", len(tfidf.get_feature_names_out()))

modelo_texto = Pipeline([("tfidf", tfidf), ("clf", LogisticRegression(max_iter=1000))])
modelo_texto.fit(ejemplo_texto["mensaje"], ejemplo_texto["intencion"])
modelo_texto.predict(["hace dos dias que no tengo internet"])

### Checklist antes de pasar a la parte 2
- [ ] Tengo `preprocesador.joblib`, `modelo.onnx` y `metadata.json`
- [ ] El ONNX pasó la verificación con `onnxruntime`
- [ ] Sé cuál es la versión de mi modelo, su framework y su umbral
- [ ] Sé cuántas filas descartó la limpieza y por qué (R1–R4)
- [ ] Puedo explicar por qué PyTorch / TensorFlow no van dentro de la imagen